# Sample Selection Algorithms

The following code contains all the 5 sample selection algorithms that were evaluated for few-shot prompting. 

The algorithms are: random, BM25, UniXcoder, E5 + FAISS, AST-normalized strutural similarity 

Each algorithm was evaluated separately to ensure the accuracy of its results. In this notebook we will run prompt 7 with each of the algorithms on 100 datapoints. This will provide the data for the LLM as Judge evaluation process to decide which algorithms works the best in our usecase. 

In [ ]:
!pip install astor
!pip install rank_bm25
# !pip install faiss
!pip install faiss-cpu

In [ ]:
import os
import time
import logging
import openai
from tqdm import tqdm
from datasets import load_dataset, Dataset, concatenate_datasets
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import ast, astor
from datasets import Dataset
import re
import random
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
import torch

# --- CONFIGURATION ---
# The dataset containing the unlabeled code snippets to analyze
UNLABELED_DATASET_PATH = "businessrules/Code_snippets"

# The dataset containing the golden examples for few-shot prompting
GOLDEN_DATASET_PATH = "businessrules/Golden_dataset"

# The number of most relevant examples to use for few-shot prompting
NUM_FEW_SHOTS = 3

# The output path for the generated business rules
OUTPUT_DATASET_PATH = "businessrules/prompt_algorithms_100"

# MODEL = "openai/gpt-4.1"
print ("done")

done


In [ ]:
import os
from openai import OpenAI

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("No OpenAI API key found. Please set OPENAI_API_KEY.")

client = OpenAI(api_key=openai_api_key)

MODEL = "gpt-4.1" 
print(f"Connected to OpenAI using model: {MODEL}")

import os
os.environ["HF_TOKEN"] = "hf_token" # removed for safety

from huggingface_hub import login
login(os.environ["HF_TOKEN"])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Connected to OpenAI using model: gpt-4.1


In [ ]:
## Prompt 7
def build_prompt(few_shot_examples, current_code):
    system_prompt = (
        "You are an expert Business Rule Extraction Model. "
        "Your goal is to identify and formalize business rules embedded in source code. "
        "You think in ordered stages: first you learn from examples, then you extract, filter, and rewrite. "
        "Your outputs must include a readable markdown section for non-technical audiences. "
        "You must silently reason through all stages internally but only output the final markdown section — "
        "never show intermediate stages, reasoning text, or any explanation."
    )

    user_prompt = f"""
You are a **Business Rule Extraction Model** trained to convert code logic into formal, human-readable **business rules**.

You will be given:
1. A few-shot set of examples (code + their corresponding business rules)
2. A new code snippet (**Code A**) to analyze

Your job is to extract all potential business rules from Code A using the following staged reasoning process.

**Important Instruction:**
You must go through all the following stages carefully *in your internal reasoning*,
but your final response must **only include the finished markdown section of business rules.**
Do **not** output or describe any intermediate stages or thoughts.

---

## STAGE 1 — LEARN FROM FEW-SHOT EXAMPLES (EVIDENCE-BASED + MARKDOWN-AWARE)

You will learn directly from the provided few-shot examples.
Each example includes:
- Source **code** (technical implementation)
- Corresponding **business rule document** written in **Markdown**

Your task is to extract learning by observing **both** the semantic relationship
between the two and the Markdown formatting patterns used.

### Instructions

For each example:
1. **Compare code ↔ business rules side-by-side.**
   - Identify terms, actions, or logic that appear in both.
   - Record only those as **confirmed correspondences**.

2. **Ignore unsupported or purely technical terms.**
   - Skip internal objects, services, managers, UI fields, and framework elements
     that never appear in the business rules.

3. **Observe Markdown structure.**
   - Note how headings, sub-sections, and lists are used.
   - Observe where emphasis (bold, italics) appears and what purpose it serves.
   - Identify consistent section patterns (e.g., “### Rule Description”, “**Condition:**”).

4. **Summarize the learning as follows:**

### LEARNED GUIDELINES (from examples)

#### 1. Confirmed Term Translation Pairs
Only include pairs that are **explicitly evidenced** in both the code and the Markdown rules.
Do **not** guess or invent pairs (e.g., never infer “entityManager → data manager”).

#### 2. Ignored / Excluded Terms and Patterns
List code patterns or elements that never map to rule text.

#### 3. Markdown Structure Patterns
Describe observed Markdown elements:
- heading levels
- list formatting
- emphasis conventions
- rule numbering or grouping styles

#### 4. Rule Style and Wording Notes
Describe how rules are expressed (imperative tone, modal verbs, grouping, clarity, etc.).

Keep this section concise (≈200–300 tokens).
Base every insight strictly on **observable evidence** from the examples.
Do **not** generate any new rules yet.

"""
    for i, ex in enumerate(few_shot_examples):
        user_prompt += f"""
### Example {i+1}
**Code:**
{ex['code']}

**Business Rules:**
{ex['rule']}
---
"""

    user_prompt += f"""
## STAGE 2 — EXTRACT CANDIDATE BEHAVIORS
From **Code A**, list every significant logical or conditional behavior in plain language,
even if it appears technical.
These are your **candidate behaviors** — raw statements of what the code seems to do.

Present them as short, clear bullet points under the heading:

**Candidate Behaviors (Raw Extracts):**

## STAGE 3 — FILTER AND REWRITE BUSINESS RULES
You are an expert in **business rule modeling**.
Now decide which candidate behaviors truly represent *business policy*, not technical or UI details.

### Exclusion Principle (high priority)
Before rewriting, **discard any candidate that only concerns user interfaces, data entry forms, field definitions, labels, widgets, or display logic.**
These include:
- Adding or configuring form fields (`->add(...)`, `ChoiceType`, `TextType`, etc.)
- Field attributes like `label`, `required`, `mapped`, `placeholder`
- Any constraints tied to UI validation (`NotBlank`, `Length`, `Choice`)
- Rendering or view variables (`buildView`, `getBlockPrefix`, `getName`, etc.)

If a candidate’s meaning exists *only* at presentation or form level, mark it **EXCLUDED** with reason “UI/Presentation Detail”.

### Rewrite Principle
For the remaining candidates:
1. Keep only behaviors that reflect **business intent**, **policy**, or **domain rules** (e.g., pricing, eligibility, payment conditions, booking states, approval flows).
2. Express them in human-readable form, following the LEARNED GUIDELINES.
3. Use modal verbs (**must**, **may**, **cannot**) and avoid system terms, APIs, or class names.
4. Merge only when it increases clarity without changing semantics.

## STAGE 4 — VERIFICATION
Perform a light verification of Stage 3 output **without rewriting text**:

- Check that all Stage 3 business rules are atomic, readable, and testable.
- Do **not** modify the rule text; just report issues if found.

## FINAL OUTPUT FORMAT

### Markdown Section with all the selected business rules
Start with a markdown heading:

**Business Rules for [name of the section we are writing business rules for]**

Then write the finalized, refined business rules in markdown format, including headings, bullets, bold, numbering, etc that make it readable.
No explanations, reasoning prose, or intermediate outputs.

### Current Code to Analyze
Code:
{current_code}

### Important Reminders:
- Follow the staged reasoning (Learn → Extract → Filter → Rewrite → Verify) before producing the final output.
- **Only show the final markdown section in your answer. Do not include any thought process, explanations, or reasoning text.**
"""
    return system_prompt, user_prompt


In [ ]:
golden_dataset = load_dataset(GOLDEN_DATASET_PATH, split="train")
print ("done")

In [ ]:
# Algorith 1: Random
def retrieve_few_shots_algorithm1(input_code: str, num_few_shots: int, pool_size: int = 50):
    """
    Algorithm 1 — Random Sampling
    Selects `num_few_shots` random few-shot examples from the first `pool_size` entries
    of the golden dataset. Used as a baseline.

    Args:
        current_code (str): The input code snippet (not used here, but kept for interface consistency).
        num_few_shots (int): Number of few-shot examples to retrieve.
        pool_size (int, optional): How many golden examples to consider before sampling. Default = 50.

    Returns:
        list[dict]: List of few-shot examples, each with 'ID', 'code', and 'rule'.
    """
    golden_data = list(golden_dataset)

    upper_limit = min(pool_size, len(golden_data))
    sample_pool = golden_data[:upper_limit]

    selected_samples = random.sample(sample_pool, num_few_shots)

    few_shots = [
        {
            "ID": item["id"],
            "code": item["cd"],
            "rule": item["br"],
        }
        for i, item in enumerate(selected_samples)
    ]

    return few_shots

In [ ]:
# Algorithm 2: BM25
CODE_STOPWORDS = {
    "def", "class", "public", "private", "protected", "return", "if", "else", "elif", "for", "while",
    "switch", "case", "break", "continue", "function", "var", "const", "let", "import", "from",
    "package", "extends", "implements", "try", "catch", "finally", "new", "this", "super", "static",
    "void", "with", "main", "args", "println", "system", "print", "log", "logger", "debug", "error",
    "do", "end", "then", "begin", "endif", "foreach", "next", "pass", "override", "abstract",
    "interface", "enum", "int", "float", "double", "char", "boolean", "string", "true", "false",
    "null", "todo", "fixme", "note", "service", "entity", "bundle", "form", "view", "build", "event",
    "listener", "configure", "controller", "array", "peymans", "label", "php", "src", "pms",
    "master", "snippet", "builder", "vars", "options",
}

DOMAIN_KEYWORDS = {
    "booking", "reservation", "guest", "hotel", "property", "payment", "invoice", "refund",
    "charge", "transaction", "pay", "notification", "email", "message", "api", "gateway",
    "availability", "automated", "deposit"
}

def code_tokenizer(code: str):
    """Tokenizes source code into meaningful words, filtering stopwords and punctuation."""
    code = re.sub(r'([a-z])([A-Z])', r'\1 \2', code)
    code = code.replace('_', ' ')
    tokens = re.findall(r'[A-Za-z][A-Za-z0-9]+', code.lower())
    tokens = [t for t in tokens if len(t) > 2 and t not in CODE_STOPWORDS]
    return tokens

def weighted_query_tokens(tokens):
    """Increases weight of domain-relevant tokens."""
    weighted = []
    for t in tokens:
        weighted.extend([t] * 3 if t in DOMAIN_KEYWORDS else [t])
    return weighted

def retrieve_few_shots_algorithm2(input_code: str, num_few_shots: int, pool_size: int = 50):
    """
    Algorithm 2 — BM25 Contextual Similarity
    Retrieves the top `num_few_shots` examples from the first `pool_size`
    golden samples based on BM25 token-level similarity.

    Args:
        current_code (str): Input code snippet.
        num_few_shots (int): Number of few-shot examples to retrieve.
        pool_size (int, optional): How many golden examples to consider. Default = 50.

    Returns:
        list[dict]: List of few-shot examples with 'ID', 'code', and 'rule'.
    """
    golden_data = list(golden_dataset)
    upper_limit = min(pool_size, len(golden_data))
    subset = golden_data[:upper_limit]

    golden_codes = [item["cd"] for item in subset]
    tokenized_corpus = [code_tokenizer(code) for code in golden_codes]
    bm25 = BM25Okapi(tokenized_corpus)

    query_tokens = weighted_query_tokens(code_tokenizer(input_code))

    scores = bm25.get_scores(query_tokens)
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)

    selected_indices = ranked_indices[:num_few_shots]
    few_shots = [
        {
            "ID": subset[i]["id"],
            "code": subset[i]["cd"],
            "rule": subset[i]["br"],
        }
        for i in selected_indices
    ]

    return few_shots

In [ ]:
# Algorithm 3: UniXcoder semantic similarity
import torch
from sentence_transformers import SentenceTransformer, util
import random, logging

model = SentenceTransformer("microsoft/unixcoder-base")

def retrieve_few_shots_algorithm3(current_code: str, num_few_shots: int, pool_size: int = 50):
    """
    Algorithm 3 — Semantic / Lexical Similarity (UniXcoder)
    Retrieves top-k few-shot examples from the golden dataset using cosine
    similarity between code embeddings.

    If embedding or similarity fails (e.g., due to long/invalid input),
    the algorithm falls back to random few-shot selection.
    """

    golden_data = list(golden_dataset)
    upper_limit = min(pool_size, len(golden_data))
    subset = golden_data[:upper_limit]

    if len(subset) == 0:
        return []

    golden_codes = [item["cd"] for item in subset]

    try:
        golden_embeddings = model.encode(
            golden_codes,
            convert_to_tensor=True,
            show_progress_bar=False,
            normalize_embeddings=True
        )

        query_embedding = model.encode(
            current_code,
            convert_to_tensor=True,
            normalize_embeddings=True
        )

        cosine_scores = util.cos_sim(query_embedding, golden_embeddings)[0]

        top_results = torch.topk(cosine_scores, k=min(num_few_shots, len(subset)))
        selected_indices = top_results.indices.tolist()

        few_shots = [
            {
                "ID": subset[i]["id"],
                "code": subset[i]["cd"],
                "rule": subset[i]["br"],
            }
            for i in selected_indices
        ]

        return few_shots

    except Exception as e:
        # --- Fallback: Return random few-shots ---
        random_indices = random.sample(range(len(subset)), k=min(num_few_shots, len(subset)))
        few_shots = [
            {
                "ID": subset[i]["id"],
                "code": subset[i]["cd"],
                "rule": subset[i]["br"],
            }
            for i in random_indices
        ]

        return few_shots


In [ ]:
# Algorithm 4: Enriched embedding similarity (E5 + FAISS)
EMBED_MODEL_NAME = "intfloat/e5-base-v2"
model = SentenceTransformer(EMBED_MODEL_NAME)


def build_enriched_text(code: str) -> str:
    """Creates an enriched textual representation of a code snippet for embedding."""
    return f"Code snippet:\n{code}\n\nDescribe what business rules or logic this code likely implements."


def retrieve_few_shots_algorithm4(input_code: str, num_few_shots: int, pool_size: int = 50):
    """
    Algorithm 4 — Enriched Embedding-Based Similarity
    Uses combined code + business rule text embeddings to retrieve top few-shot examples.

    Args:
        current_code (str): Input code snippet.
        num_few_shots (int): Number of few-shot examples to retrieve.
        pool_size (int, optional): Number of golden examples to consider. Default = 50.

    Returns:
        list[dict]: List of few-shot examples with 'ID', 'code', and 'rule'.
    """
    golden_data = list(golden_dataset)
    upper_limit = min(pool_size, len(golden_data))
    subset = golden_data[:upper_limit]

    enriched_texts = [
        f"Code snippet:\n{item['cd']}\n\nBusiness rule summary:\n{item['br']}"
        for item in subset
    ]

    golden_embeddings = model.encode(enriched_texts, convert_to_tensor=True, show_progress_bar=False)

    enriched_query = build_enriched_text(input_code)
    query_emb = model.encode(enriched_query, convert_to_tensor=True)

    cos_scores = util.cos_sim(query_emb, golden_embeddings)[0]
    topk = torch.topk(cos_scores, k=min(num_few_shots, cos_scores.shape[0]))

    selected_indices = topk.indices.tolist()
    few_shots = [
        {
            "ID": subset[i]["id"],
            "code": subset[i]["cd"],
            "rule": subset[i]["br"],
        }
        for i in selected_indices
    ]

    return few_shots

In [ ]:
import torch
import random
import logging
from sentence_transformers import util

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


def _clean_text_for_embed(s: str, max_chars: int = 2000) -> str:
    """Sanitize text for embedding: ensure string, remove nulls, escape dangerous sequences, truncate."""
    if not isinstance(s, str):
        return ""
    cleaned = s.encode("unicode_escape").decode("utf-8")
    cleaned = cleaned.replace("\x00", "")  
    cleaned = cleaned.replace("\r\n", "\n").replace("\r", "\n")
    if len(cleaned) > max_chars:
        cleaned = cleaned[:max_chars]
    return cleaned

def retrieve_few_shots_algorithm4(input_code: str, num_few_shots: int, pool_size: int = 50, max_chars: int = 2000):
    """
    Robust Algorithm 4 — Enriched Embedding-Based Similarity with safe encoding and random fallback.
    """
    try:
        golden_data = list(golden_dataset)
    except Exception as e:
        logging.error(f"[Alg4] Could not read golden_dataset: {e}")
        return []

    if len(golden_data) == 0:
        logging.error("[Alg4] golden_dataset is empty.")
        return []

    upper_limit = min(pool_size, len(golden_data))
    subset = golden_data[:upper_limit]

    enriched_texts = []
    valid_idx_map = []  

    for idx, item in enumerate(subset):
        code_part = item.get("cd", "")
        rule_part = item.get("br", "")
        if not isinstance(code_part, str):
            continue
        enriched = f"Code snippet:\n{code_part}\n\nBusiness rule summary:\n{rule_part}"
        cleaned = _clean_text_for_embed(enriched, max_chars=max_chars)
        if not cleaned:
            continue
        enriched_texts.append(cleaned)
        valid_idx_map.append(idx)

    if len(enriched_texts) == 0:
        logging.error("[Alg4] No valid enriched texts to encode; falling back to random samples.")
        random_indices = random.sample(range(len(subset)), k=min(num_few_shots, len(subset)))
        return [
            {"ID": subset[i]["id"], "code": subset[i]["cd"], "rule": subset[i]["br"]}
            for i in random_indices
        ]

    encoded_list = []
    for i, txt in enumerate(enriched_texts):
        try:
            emb = model.encode(txt, convert_to_tensor=True, show_progress_bar=False, normalize_embeddings=True)
            encoded_list.append(emb)
        except Exception as e:
            logging.error(f"[Alg4] Error encoding candidate enriched_text index {i} (subset idx {valid_idx_map[i]}): {e}")
            continue

    if len(encoded_list) == 0:
        random_indices = random.sample(range(len(subset)), k=min(num_few_shots, len(subset)))
        return [
            {"ID": subset[i]["id"], "code": subset[i]["cd"], "rule": subset[i]["br"]}
            for i in random_indices
        ]

    # Stack embeddings into tensor
    try:
        golden_embeddings = torch.stack(encoded_list)
    except Exception as e:
        logging.error(f"[Alg4] Failed to stack embeddings: {e}. Falling back to random.")
        random_indices = random.sample(range(len(subset)), k=min(num_few_shots, len(subset)))
        return [
            {"ID": subset[i]["id"], "code": subset[i]["cd"], "rule": subset[i]["br"]}
            for i in random_indices
        ]

    cleaned_query = _clean_text_for_embed(build_enriched_text(input_code), max_chars=max_chars)
    try:
        query_emb = model.encode(cleaned_query, convert_to_tensor=True, normalize_embeddings=True)
    except Exception as e:
        random_indices = random.sample(range(len(subset)), k=min(num_few_shots, len(subset)))
        return [
            {"ID": subset[i]["id"], "code": subset[i]["cd"], "rule": subset[i]["br"]}
            for i in random_indices
        ]

    try:
        cos_scores = util.cos_sim(query_emb, golden_embeddings)[0]
    except Exception as e:
        random_indices = random.sample(range(len(subset)), k=min(num_few_shots, len(subset)))
        return [
            {"ID": subset[i]["id"], "code": subset[i]["cd"], "rule": subset[i]["br"]}
            for i in random_indices
        ]

    k = min(num_few_shots, cos_scores.shape[0])
    if k == 0:
        random_indices = random.sample(range(len(subset)), k=min(num_few_shots, len(subset)))
        return [
            {"ID": subset[i]["id"], "code": subset[i]["cd"], "rule": subset[i]["br"]}
            for i in random_indices
        ]

    try:
        topk = torch.topk(cos_scores, k=k)
        picked_positions = topk.indices.tolist()  
    except Exception as e:
        random_indices = random.sample(range(len(subset)), k=min(num_few_shots, len(subset)))
        return [
            {"ID": subset[i]["id"], "code": subset[i]["cd"], "rule": subset[i]["br"]}
            for i in random_indices
        ]

    selected_subset_indices = [valid_idx_map[pos] for pos in picked_positions if pos < len(valid_idx_map)]

    few_shots = []
    for idx in selected_subset_indices:
        try:
            few_shots.append({
                "ID": subset[idx].get("id", f"unknown_{idx}"),
                "code": subset[idx].get("cd", ""),
                "rule": subset[idx].get("br", ""),
            })
        except Exception as e:
            logging.error(f"[Alg4] Error building few-shot for subset idx {idx}: {e}")
            continue

    if len(few_shots) < num_few_shots:
        deficit = num_few_shots - len(few_shots)
        remaining_idxs = [i for i in range(len(subset)) if subset[i]["id"] not in {fs["ID"] for fs in few_shots}]
        if remaining_idxs:
            add_idxs = random.sample(remaining_idxs, k=min(deficit, len(remaining_idxs)))
            for ai in add_idxs:
                few_shots.append({
                    "ID": subset[ai].get("id", f"unknown_{ai}"),
                    "code": subset[ai].get("cd", ""),
                    "rule": subset[ai].get("br", ""),
                })

    return few_shots


In [ ]:
# Algorithm 5: AST-NORMALIZEd structural similarity (CodeBERT + FAISS)
class NormalizeNames(ast.NodeTransformer):
    """
    Normalizes variable, function, and class names in an AST to focus on code structure
    rather than naming conventions.
    """
    def __init__(self):
        super().__init__()
        self.var_map = {}
        self.counter = 0

    def _new_name(self, base="var"):
        self.counter += 1
        return f"{base}{self.counter}"

    def visit_Name(self, node):
        if node.id not in self.var_map:
            self.var_map[node.id] = self._new_name()
        node.id = self.var_map[node.id]
        return node

    def visit_arg(self, node):
        if node.arg not in self.var_map:
            self.var_map[node.arg] = self._new_name("arg")
        node.arg = self.var_map[node.arg]
        return node

    def visit_FunctionDef(self, node):
        node.name = "func"
        self.generic_visit(node)
        return node

    def visit_ClassDef(self, node):
        node.name = "Class"
        self.generic_visit(node)
        return node


def normalize_code(code_str: str) -> str:
    """Converts code into a normalized AST representation."""
    try:
        tree = ast.parse(code_str)
        normalizer = NormalizeNames()
        normalized_tree = normalizer.visit(tree)
        return astor.to_source(normalized_tree)
    except Exception:
        return code_str

golden_data = list(golden_dataset)
golden_codes = [item["cd"] for item in golden_data]

embed_model = SentenceTransformer("microsoft/codebert-base")

normalized_codes = [normalize_code(code) for code in golden_codes]
embeddings = embed_model.encode(normalized_codes, convert_to_numpy=True, normalize_embeddings=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)


def retrieve_few_shots_algorithm5(input_code: str, num_few_shots: int, pool_size: int = 50):
    """
    Algorithm 5 — AST-normalized structural similarity (GraphCodeBERT + FAISS).

    Focuses on structural similarity by normalizing code identifiers,
    embedding via GraphCodeBERT, and retrieving nearest neighbors using FAISS.

    Args:
        current_code (str): Input code snippet to retrieve few-shots for.
        num_few_shots (int): Number of examples to retrieve.
        pool_size (int): Pool size to limit search (default 50).

    Returns:
        list[dict]: Few-shot examples with 'ID', 'code', and 'rule'.
    """
    query_norm = normalize_code(input_code)

    query_emb = embed_model.encode([query_norm], convert_to_numpy=True, normalize_embeddings=True)
    scores, idxs = index.search(query_emb, k=min(num_few_shots, len(golden_data)))

    top_indices = [i for i in idxs[0] if i < pool_size][:num_few_shots]

    few_shots = [
        {
            "ID": golden_data[i]["id"],
            "code": golden_data[i]["cd"],
            "rule": golden_data[i]["br"]
        }
        for i in top_indices
    ]

    return few_shots

# Running all 5 algorithms on 100 datapoints
Prepare the data for LLM as Judge evaluation to decide which sampling algorithm is best for our usecase. 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Use persistent storage path
LOCAL_CHECKPOINT_PATH = "/content/drive/MyDrive/eval_checkpoints/eval_progress2.jsonl"
BASE_DIR = "/content/drive/MyDrive/openai_batches"  


In [ ]:
import pandas as pd
LOCAL_CSV_PATH = "/content/drive/MyDrive/eval_checkpoints/eval_progress100.csv"

LOCAL_JSONL_PATH = "/content/checkpoints/batch_requests100.jsonl"
os.makedirs(os.path.dirname(LOCAL_CSV_PATH), exist_ok=True)
os.makedirs(os.path.dirname(LOCAL_JSONL_PATH), exist_ok=True)


NUM_FEW_SHOTS = 3
RETRIES = 3
BATCH_SIZE = 10  # number of items per batch
LOCAL_CHECKPOINT_PATH = "/content/checkpoints/eval_progress100.jsonl"

RETRIEVERS = [
    ("result_1", retrieve_few_shots_algorithm1),
    ("result_2", retrieve_few_shots_algorithm2),
    ("result_3", retrieve_few_shots_algorithm3),
    ("result_4", retrieve_few_shots_algorithm4),
    ("result_5", retrieve_few_shots_algorithm5),
]

logging.info(f"Loading unlabeled dataset: {UNLABELED_DATASET_PATH}")
unlabeled_dataset = load_dataset(UNLABELED_DATASET_PATH, split="train")
unlabeled_dataset = unlabeled_dataset.select(range(100))  


if os.path.exists(LOCAL_CSV_PATH):
    df_prev = pd.read_csv(LOCAL_CSV_PATH)
    processed_ids = set(df_prev["ID"])
else:
    df_prev = pd.DataFrame()
    processed_ids = set()

try:
    existing_dataset = load_dataset(OUTPUT_DATASET_PATH, split="train")
    processed_ids.update(existing_dataset["ID"])
except Exception:
    existing_dataset = None

pending_items = [item for item in unlabeled_dataset if item["ID"] not in processed_ids]
print(f"{len(pending_items)} items pending out of {len(unlabeled_dataset)} total.")

def build_batch_requests(batch):
    requests = []
    for item in batch:
        file_id = item["ID"]
        current_code = item["txt_file"]
        for result_name, retriever_func in RETRIEVERS:
            try:
                few_shots = retriever_func(current_code, NUM_FEW_SHOTS)
                system_prompt, user_prompt = build_prompt(few_shots, current_code)
                req = {
                    "custom_id": f"{file_id}_{result_name}",
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": MODEL,
                        "messages": [
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_prompt},
                        ],
                        "temperature": 0.2,
                        "max_tokens": 4000,
                    },
                }
                requests.append(req)
            except Exception as e:
                logging.error(f"Failed to build request for {file_id} ({result_name}): {e}")
    return requests

100 items pending out of 100 total.


In [ ]:
import json
# --- MAIN LOOP ---
for i in range(0, len(pending_items), BATCH_SIZE):
    batch = pending_items[i : i + BATCH_SIZE]
    requests = build_batch_requests(batch)

    with open(LOCAL_JSONL_PATH, "w") as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")

    upload_resp = client.files.create(file=open(LOCAL_JSONL_PATH, "rb"), purpose="batch")
    file_id = upload_resp.id
    print(f"Uploaded batch file (id={file_id})")

    batch_job = client.batches.create(
        input_file_id=file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    print(f"Submitted batch {i//BATCH_SIZE + 1}, ID: {batch_job.id}")
    print("Track progress: https://platform.openai.com/batches")

    while True:
        status = client.batches.retrieve(batch_job.id)
        if status.status == "completed":
            print(f" Batch {batch_job.id} completed.")
            break
        elif status.status in ["failed", "cancelled"]:
            print(f" Batch {batch_job.id} failed with status: {status.status}")
            break
        else:
            print(f"Batch running... ({status.status})")
            time.sleep(60)

    if status.status == "completed":
        result_file = client.files.content(status.output_file_id)
        local_output = f"{BASE_DIR}/batch_output_{i//BATCH_SIZE+1}.jsonl"
        with open(local_output, "wb") as f:
            f.write(result_file.read())

        results = []
        with open(local_output, "r") as f:
            for line in f:
                data = json.loads(line)
                if "response" in data:
                    cid = data["custom_id"]
                    file_id, result_name = cid.split("_", 1)
                    output_text = data["response"]["body"]["choices"][0]["message"]["content"]
                    results.append({"ID": file_id, result_name: output_text})

        df_new = pd.DataFrame(results)
        df_pivot = df_new.pivot_table(index="ID", values=["result_2", "result_3", "result_4", "result_5"], aggfunc="first").reset_index()

        if os.path.exists(LOCAL_CSV_PATH):
            df_prev = pd.read_csv(LOCAL_CSV_PATH)

            df_prev["ID"] = df_prev["ID"].astype(str)
            df_pivot["ID"] = df_pivot["ID"].astype(str)

            df_combined = pd.merge(df_prev, df_pivot, on="ID", how="outer")
        else:
            df_combined = df_pivot
      
        df_combined.drop_duplicates(subset=["ID"], keep="last").to_csv(LOCAL_CSV_PATH, index=False)
        print(f" Progress saved locally to {LOCAL_CSV_PATH}")

        try:
            new_entries = Dataset.from_pandas(df_combined)
            try:
                remote_dataset = load_dataset(OUTPUT_DATASET_PATH, split="train")
                combined = concatenate_datasets([remote_dataset, new_entries])
            except Exception:
                combined = new_entries

            df_final = combined.to_pandas().drop_duplicates(subset=["ID"], keep="last").reset_index(drop=True)
            combined = Dataset.from_pandas(df_final)
            combined.push_to_hub(OUTPUT_DATASET_PATH)
            print(f"  Pushed batch {i//BATCH_SIZE + 1} to HF dataset.")
        except Exception as e:
            logging.error(f"Failed to push batch {i//BATCH_SIZE + 1}: {e}")
            time.sleep(5)

print(" All batches submitted, processed, and saved successfully.")

In [ ]:
BASE_DIR = "/content/drive/MyDrive/openai_batches"  
os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
import os
import time
import json
import pandas as pd

# === Setup Paths ===
LOCAL_CSV_PATH = "/content/drive/MyDrive/eval_checkpoints/eval_progress1002.csv"
LOCAL_JSONL_PATH = "/content/checkpoints/batch_requests1002.jsonl"
BATCH_RECORDS_PATH = "/content/drive/MyDrive/eval_checkpoints/batch_records.json"

os.makedirs(os.path.dirname(LOCAL_CSV_PATH), exist_ok=True)
os.makedirs(os.path.dirname(LOCAL_JSONL_PATH), exist_ok=True)

NUM_FEW_SHOTS = 3
RETRIES = 3
BATCH_SIZE = 10

RETRIEVERS = [
    ("result_1", retrieve_few_shots_algorithm1),
    ("result_2", retrieve_few_shots_algorithm2),
    ("result_3", retrieve_few_shots_algorithm3),
    ("result_4", retrieve_few_shots_algorithm4),
    ("result_5", retrieve_few_shots_algorithm5),
]

# === Load Unlabeled Data ===
logging.info(f"Loading unlabeled dataset: {UNLABELED_DATASET_PATH}")
unlabeled_dataset = load_dataset(UNLABELED_DATASET_PATH, split="train")
unlabeled_dataset = unlabeled_dataset.select(range(40, 100))

# === Filter Already Processed ===
if os.path.exists(LOCAL_CSV_PATH):
    df_prev = pd.read_csv(LOCAL_CSV_PATH)
    processed_ids = set(df_prev["ID"])
else:
    processed_ids = set()

try:
    existing_dataset = load_dataset(OUTPUT_DATASET_PATH, split="train")
    processed_ids.update(existing_dataset["ID"])
except Exception:
    pass

pending_items = [item for item in unlabeled_dataset if item["ID"] not in processed_ids]
print(f"{len(pending_items)} items pending out of {len(unlabeled_dataset)} total.")

# === Build Requests ===
def build_batch_requests(batch):
    requests = []
    for item in batch:
        file_id = item["ID"]
        current_code = item["txt_file"]
        for result_name, retriever_func in RETRIEVERS:
            try:
                few_shots = retriever_func(current_code, NUM_FEW_SHOTS)
                system_prompt, user_prompt = build_prompt(few_shots, current_code)
                req = {
                    "custom_id": f"{file_id}_{result_name}",
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": MODEL,
                        "messages": [
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_prompt},
                        ],
                        "temperature": 0.2,
                        "max_tokens": 4000,
                    },
                }
                requests.append(req)
            except Exception as e:
                logging.error(f"Failed to build request for {file_id} ({result_name}): {e}")
    return requests

# === Submit Batches ===
batch_records = []

for i in range(0, len(pending_items), BATCH_SIZE):
    batch = pending_items[i : i + BATCH_SIZE]
    requests = build_batch_requests(batch)

    with open(LOCAL_JSONL_PATH, "w") as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")

    upload_resp = client.files.create(file=open(LOCAL_JSONL_PATH, "rb"), purpose="batch")
    file_id = upload_resp.id
    print(f"Uploaded batch file (id={file_id})")

    batch_job = client.batches.create(
        input_file_id=file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

    print(f"Submitted batch {i//BATCH_SIZE + 1}, ID: {batch_job.id}")
    print("Track progress at: https://platform.openai.com/batches")

    batch_records.append({
        "batch_number": i // BATCH_SIZE + 1,
        "batch_id": batch_job.id,
        "input_file_id": file_id,
        "status": "submitted"
    })

    while True:
        status = client.batches.retrieve(batch_job.id)
        if status.status == "completed":
            print(f" Batch {batch_job.id} completed.")
            batch_records[-1]["status"] = "completed"
            batch_records[-1]["output_file_id"] = status.output_file_id
            break
        elif status.status in ["failed", "cancelled"]:
            print(f" Batch {batch_job.id} failed ({status.status})")
            batch_records[-1]["status"] = status.status
            break
        else:
            print(f"Batch running... ({status.status})")
            time.sleep(60)

# === Save Batch Records ===
with open(BATCH_RECORDS_PATH, "w") as f:
    json.dump(batch_records, f, indent=2)

print(" All batches submitted and tracked.")
print(f" Saved batch record file: {BATCH_RECORDS_PATH}")

# Display the collected IDs
print("\nBatch IDs:")
for rec in batch_records:
    print(f"{rec['batch_number']:02d}: {rec['batch_id']}")


In [ ]:
# Getting the batch results from OpenAI
from openai import OpenAI
import pandas as pd
from datasets import Dataset

# --- 1️ Initialize client ---
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# --- 2️ Your batch IDs ---
batch_ids = [
    "batch_690bc27b296081908a343e55330c1ce9",
    "batch_690bc72ef528819091c97aa71820d921",
    "batch_690bcaefe614819092780d915584cf01",
    "batch_690bcf267ec88190a45d158617b635e2",
    "batch_690c540e5064819081ccc1a5fd83a1dd",
    "batch_690c596255e88190bbb6cee51f31a60f",
    "batch_690c5f2ed8b88190860b3ce6610b635c",
    "batch_690c64be875481908f7a2c513f0c6db9",
    "batch_690c6a0a47fc81909d48ad68c86ef321",
    "batch_690c7393f26081909fda91c9ee038bfd"
]


# --- 3️ Function to download each batch result from OpenAI ---
def download_batch_result(batch_id):
    try:
        batch = client.batches.retrieve(batch_id)
        if not getattr(batch, "output_file_id", None):
            print(f" No output file yet for batch {batch_id}")
            return None

        output_file = client.files.content(batch.output_file_id)
        df = pd.read_json(output_file.text, lines=True)  # Batch outputs are JSONL
        df["batch_id"] = batch_id  # Track origin
        print(f" Downloaded batch {batch_id} ({len(df)} rows)")
        return df

    except Exception as e:
        print(f" Failed to download batch {batch_id}: {e}")
        return None

# --- 4️ Download all batches ---
batch_dfs = []
for bid in batch_ids:
    df = download_batch_result(bid)
    if df is not None:
        batch_dfs.append(df)

if not batch_dfs:
    raise ValueError("No batch data downloaded.")

combined_df = pd.concat(batch_dfs, ignore_index=True)
print(f"\n Combined {len(combined_df)} rows from {len(batch_dfs)} batches")

# --- 5️ Parse nested content ---
def parse_batch_results(df):
    parsed = []
    for _, row in df.iterrows():
        try:
            code_id, result_name = row["custom_id"].split("_", 1)
            response = row["response"]
            if isinstance(response, str):
                response = json.loads(response)

            # Safely extract content
            content = (
                response["body"]["choices"][0]["message"]["content"]
                if "body" in response and "choices" in response["body"]
                else None
            )

            parsed.append({
                "ID": code_id,
                result_name: content
            })
        except Exception as e:
            print(f" Failed to parse row {row.get('custom_id')}: {e}")

    df_parsed = pd.DataFrame(parsed)
    # Group by ID so each row shows result_1…result_5
    df_grouped = df_parsed.groupby("ID").first().reset_index()
    return df_grouped

df_grouped = parse_batch_results(combined_df)

# --- 6️ Save locally ---
output_csv = "/content/merged_batches.csv"
df_grouped.to_csv(output_csv, index=False)
print(f" Merged batch results saved to {output_csv}")

# --- 7️ Push to Hugging Face (optional) ---
try:
    hf_dataset = Dataset.from_pandas(df_grouped)
    hf_dataset.push_to_hub("businessrules/100_5algorithms")
    print(" Successfully pushed dataset to Hugging Face!")
except Exception as e:
    print(f" Hugging Face push failed: {e}")




In [ ]:
drive_folder = "/content/drive/MyDrive/algorithms"
os.makedirs(drive_folder, exist_ok=True)

output_csv = os.path.join(drive_folder, "merged_batches.csv")
df_grouped.to_csv(output_csv, index=False)
print(f" Merged batch results saved to {output_csv}")